In [1]:
 %%capture
!pip install transformers>=4.41.2 sentence-transformers>=3.0.1 gensim>=4.3.2 scikit-learn>=1.5.0 accelerate>=0.31.0

# Downloading and Running An LLM

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately and keep them as such so that we can explore them separately.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [3]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"

# Tokenize the input prompt
input_ids = tokenizer(prompt,return_tensors="pt").input_ids.to("cuda")

# Generate the text
generation_output = model.generate(
    input_ids=input_ids,
    max_new_tokens=20
)
# Print the output
print(tokenizer.decode(generation_output[0]))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|> Subject: Heartfelt Apologies for the Gardening Mishap


Dear


In [4]:
print(input_ids)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001]], device='cuda:0')


In [5]:
for id in input_ids[0]:
   print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>


In [6]:
generation_output

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001,  3323,   622, 29901, 17778, 29888,  2152,
          6225, 11763,   363,   278, 19906,   292,   341,   728,   481,    13,
            13,    13, 29928,   799]], device='cuda:0')

In [7]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

Sub
ject
Subject
:


# Comparing Trained LLM Tokenizers


In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer

colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
  tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
  tokens_ids = tokenizer(sentence).input_ids
  for idx, t in enumerate(tokens_ids):
    print(
        f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}]m' +
        tokenizer.decode(t) +
        '\x1b[0m',
        end=' '
    )

In [11]:
text = """
English and CAPITALIZATION
🎵 🐦
show_tokens False None elif == >= else: two tabs:"  " Three tabs: "   "
12.0*50=600
"""

In [12]:
show_tokens(text, "bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[0;30;48;2;102;194;165]m[CLS] [0;30;48;2;252;141;98]menglish [0;30;48;2;141;160;203]mand [0;30;48;2;231;138;195]mcapital [0;30;48;2;166;216;84]m##ization [0;30;48;2;255;217;47]m[UNK] [0;30;48;2;102;194;165]m[UNK] [0;30;48;2;252;141;98]mshow [0;30;48;2;141;160;203]m_ [0;30;48;2;231;138;195]mtoken [0;30;48;2;166;216;84]m##s [0;30;48;2;255;217;47]mfalse [0;30;48;2;102;194;165]mnone [0;30;48;2;252;141;98]meli [0;30;48;2;141;160;203]m##f [0;30;48;2;231;138;195]m= [0;30;48;2;166;216;84]m= [0;30;48;2;255;217;47]m> [0;30;48;2;102;194;165]m= [0;30;48;2;252;141;98]melse [0;30;48;2;141;160;203]m: [0;30;48;2;231;138;195]mtwo [0;30;48;2;166;216;84]mtab [0;30;48;2;255;217;47]m##s [0;30;48;2;102;194;165]m: [0;30;48;2;252;141;98]m" [0;30;48;2;141;160;203]m" [0;30;48;2;231;138;195]mthree [0;30;48;2;166;216;84]mtab [0;30;48;2;255;217;47]m##s [0;30;48;2;102;194;165]m: [0;30;48;2;252;141;98]m" [0;30;48;2;141;160;203]m" [0;30;48;2;231;138;195]m12 [0;30;48;2;166;216;84]m. 

In [13]:
show_tokens(text, "bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[0;30;48;2;102;194;165]m[CLS] [0;30;48;2;252;141;98]mEnglish [0;30;48;2;141;160;203]mand [0;30;48;2;231;138;195]mCA [0;30;48;2;166;216;84]m##PI [0;30;48;2;255;217;47]m##TA [0;30;48;2;102;194;165]m##L [0;30;48;2;252;141;98]m##I [0;30;48;2;141;160;203]m##Z [0;30;48;2;231;138;195]m##AT [0;30;48;2;166;216;84]m##ION [0;30;48;2;255;217;47]m[UNK] [0;30;48;2;102;194;165]m[UNK] [0;30;48;2;252;141;98]mshow [0;30;48;2;141;160;203]m_ [0;30;48;2;231;138;195]mtoken [0;30;48;2;166;216;84]m##s [0;30;48;2;255;217;47]mF [0;30;48;2;102;194;165]m##als [0;30;48;2;252;141;98]m##e [0;30;48;2;141;160;203]mNone [0;30;48;2;231;138;195]mel [0;30;48;2;166;216;84]m##if [0;30;48;2;255;217;47]m= [0;30;48;2;102;194;165]m= [0;30;48;2;252;141;98]m> [0;30;48;2;141;160;203]m= [0;30;48;2;231;138;195]melse [0;30;48;2;166;216;84]m: [0;30;48;2;255;217;47]mtwo [0;30;48;2;102;194;165]mta [0;30;48;2;252;141;98]m##bs [0;30;48;2;141;160;203]m: [0;30;48;2;231;138;195]m" [0;30;48;2;166;216;84]m" 

In [16]:
show_tokens(text, "gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[0;30;48;2;102;194;165]m
 [0;30;48;2;252;141;98]mEnglish [0;30;48;2;141;160;203]m and [0;30;48;2;231;138;195]m CAP [0;30;48;2;166;216;84]mITAL [0;30;48;2;255;217;47]mIZ [0;30;48;2;102;194;165]mATION [0;30;48;2;252;141;98]m
 [0;30;48;2;141;160;203]m� [0;30;48;2;231;138;195]m� [0;30;48;2;166;216;84]m� [0;30;48;2;255;217;47]m � [0;30;48;2;102;194;165]m� [0;30;48;2;252;141;98]m� [0;30;48;2;141;160;203]m
 [0;30;48;2;231;138;195]mshow [0;30;48;2;166;216;84]m_ [0;30;48;2;255;217;47]mt [0;30;48;2;102;194;165]mok [0;30;48;2;252;141;98]mens [0;30;48;2;141;160;203]m False [0;30;48;2;231;138;195]m None [0;30;48;2;166;216;84]m el [0;30;48;2;255;217;47]mif [0;30;48;2;102;194;165]m == [0;30;48;2;252;141;98]m >= [0;30;48;2;141;160;203]m else [0;30;48;2;231;138;195]m: [0;30;48;2;166;216;84]m two [0;30;48;2;255;217;47]m tabs [0;30;48;2;102;194;165]m:" [0;30;48;2;252;141;98]m  [0;30;48;2;141;160;203]m " [0;30;48;2;231;138;195]m Three [0;30;48;2;166;216;84]m tabs [0;30;

In [17]:
show_tokens(text, "google/flan-t5-small")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

[0;30;48;2;102;194;165]mEnglish [0;30;48;2;252;141;98]mand [0;30;48;2;141;160;203]mCA [0;30;48;2;231;138;195]mPI [0;30;48;2;166;216;84]mTAL [0;30;48;2;255;217;47]mIZ [0;30;48;2;102;194;165]mATION [0;30;48;2;252;141;98]m [0;30;48;2;141;160;203]m<unk> [0;30;48;2;231;138;195]m [0;30;48;2;166;216;84]m<unk> [0;30;48;2;255;217;47]mshow [0;30;48;2;102;194;165]m_ [0;30;48;2;252;141;98]mto [0;30;48;2;141;160;203]mken [0;30;48;2;231;138;195]ms [0;30;48;2;166;216;84]mFal [0;30;48;2;255;217;47]ms [0;30;48;2;102;194;165]me [0;30;48;2;252;141;98]mNone [0;30;48;2;141;160;203]m [0;30;48;2;231;138;195]me [0;30;48;2;166;216;84]ml [0;30;48;2;255;217;47]mif [0;30;48;2;102;194;165]m= [0;30;48;2;252;141;98]m= [0;30;48;2;141;160;203]m> [0;30;48;2;231;138;195]m= [0;30;48;2;166;216;84]melse [0;30;48;2;255;217;47]m: [0;30;48;2;102;194;165]mtwo [0;30;48;2;252;141;98]mtab [0;30;48;2;141;160;203]ms [0;30;48;2;231;138;195]m: [0;30;48;2;166;216;84]m" [0;30;48;2;255;217;47]m" [0;3

In [18]:
# The official is `tiktoken` but this the same tokenizer on the HF platform
show_tokens(text, "Xenova/gpt-4")

tokenizer_config.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[0;30;48;2;102;194;165]m
 [0;30;48;2;252;141;98]mEnglish [0;30;48;2;141;160;203]m and [0;30;48;2;231;138;195]m CAPITAL [0;30;48;2;166;216;84]mIZATION [0;30;48;2;255;217;47]m
 [0;30;48;2;102;194;165]m� [0;30;48;2;252;141;98]m� [0;30;48;2;141;160;203]m� [0;30;48;2;231;138;195]m � [0;30;48;2;166;216;84]m� [0;30;48;2;255;217;47]m� [0;30;48;2;102;194;165]m
 [0;30;48;2;252;141;98]mshow [0;30;48;2;141;160;203]m_ [0;30;48;2;231;138;195]mtokens [0;30;48;2;166;216;84]m False [0;30;48;2;255;217;47]m None [0;30;48;2;102;194;165]m elif [0;30;48;2;252;141;98]m == [0;30;48;2;141;160;203]m >= [0;30;48;2;231;138;195]m else [0;30;48;2;166;216;84]m: [0;30;48;2;255;217;47]m two [0;30;48;2;102;194;165]m tabs [0;30;48;2;252;141;98]m:" [0;30;48;2;141;160;203]m  [0;30;48;2;231;138;195]m " [0;30;48;2;166;216;84]m Three [0;30;48;2;255;217;47]m tabs [0;30;48;2;102;194;165]m: [0;30;48;2;252;141;98]m " [0;30;48;2;141;160;203]m   [0;30;48;2;231;138;195]m " [0;30;48;2;166;216;84]m


In [19]:
show_tokens(text, "bigcode/starcoder2-15b")


config.json:   0%|          | 0.00/803 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[0;30;48;2;102;194;165]m
 [0;30;48;2;252;141;98]mEnglish [0;30;48;2;141;160;203]m and [0;30;48;2;231;138;195]m CAPITAL [0;30;48;2;166;216;84]mIZATION [0;30;48;2;255;217;47]m
 [0;30;48;2;102;194;165]m� [0;30;48;2;252;141;98]m� [0;30;48;2;141;160;203]m� [0;30;48;2;231;138;195]m � [0;30;48;2;166;216;84]m� [0;30;48;2;255;217;47]m
 [0;30;48;2;102;194;165]mshow [0;30;48;2;252;141;98]m_ [0;30;48;2;141;160;203]mtokens [0;30;48;2;231;138;195]m False [0;30;48;2;166;216;84]m None [0;30;48;2;255;217;47]m elif [0;30;48;2;102;194;165]m == [0;30;48;2;252;141;98]m >= [0;30;48;2;141;160;203]m else [0;30;48;2;231;138;195]m: [0;30;48;2;166;216;84]m two [0;30;48;2;255;217;47]m tabs [0;30;48;2;102;194;165]m:" [0;30;48;2;252;141;98]m  [0;30;48;2;141;160;203]m " [0;30;48;2;231;138;195]m Three [0;30;48;2;166;216;84]m tabs [0;30;48;2;255;217;47]m: [0;30;48;2;102;194;165]m " [0;30;48;2;252;141;98]m   [0;30;48;2;141;160;203]m " [0;30;48;2;231;138;195]m
 [0;30;48;2;166;216;84]m1

In [20]:
show_tokens(text, "facebook/galactica-1.3b")

config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.00 [00:00<?, ?B/s]

[0;30;48;2;102;194;165]m
 [0;30;48;2;252;141;98]mEnglish [0;30;48;2;141;160;203]m and [0;30;48;2;231;138;195]m CAP [0;30;48;2;166;216;84]mITAL [0;30;48;2;255;217;47]mIZATION [0;30;48;2;102;194;165]m
 [0;30;48;2;252;141;98]m� [0;30;48;2;141;160;203]m� [0;30;48;2;231;138;195]m� [0;30;48;2;166;216;84]m� [0;30;48;2;255;217;47]m  [0;30;48;2;102;194;165]m� [0;30;48;2;252;141;98]m� [0;30;48;2;141;160;203]m� [0;30;48;2;231;138;195]m� [0;30;48;2;166;216;84]m
 [0;30;48;2;255;217;47]mshow [0;30;48;2;102;194;165]m_ [0;30;48;2;252;141;98]mtokens [0;30;48;2;141;160;203]m False [0;30;48;2;231;138;195]m None [0;30;48;2;166;216;84]m elif [0;30;48;2;255;217;47]m  [0;30;48;2;102;194;165]m== [0;30;48;2;252;141;98]m  [0;30;48;2;141;160;203]m> [0;30;48;2;231;138;195]m= [0;30;48;2;166;216;84]m else [0;30;48;2;255;217;47]m: [0;30;48;2;102;194;165]m two [0;30;48;2;252;141;98]m t [0;30;48;2;141;160;203]mabs [0;30;48;2;231;138;195]m: [0;30;48;2;166;216;84]m" [0;30;48;2;255;217

In [21]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

[0;30;48;2;102;194;165]m [0;30;48;2;252;141;98]m
 [0;30;48;2;141;160;203]mEnglish [0;30;48;2;231;138;195]mand [0;30;48;2;166;216;84]mC [0;30;48;2;255;217;47]mAP [0;30;48;2;102;194;165]mIT [0;30;48;2;252;141;98]mAL [0;30;48;2;141;160;203]mIZ [0;30;48;2;231;138;195]mATION [0;30;48;2;166;216;84]m
 [0;30;48;2;255;217;47]m� [0;30;48;2;102;194;165]m� [0;30;48;2;252;141;98]m� [0;30;48;2;141;160;203]m� [0;30;48;2;231;138;195]m [0;30;48;2;166;216;84]m� [0;30;48;2;255;217;47]m� [0;30;48;2;102;194;165]m� [0;30;48;2;252;141;98]m� [0;30;48;2;141;160;203]m
 [0;30;48;2;231;138;195]mshow [0;30;48;2;166;216;84]m_ [0;30;48;2;255;217;47]mto [0;30;48;2;102;194;165]mkens [0;30;48;2;252;141;98]mFalse [0;30;48;2;141;160;203]mNone [0;30;48;2;231;138;195]melif [0;30;48;2;166;216;84]m== [0;30;48;2;255;217;47]m>= [0;30;48;2;102;194;165]melse [0;30;48;2;252;141;98]m: [0;30;48;2;141;160;203]mtwo [0;30;48;2;231;138;195]mtabs [0;30;48;2;166;216;84]m:" [0;30;48;2;255;217;47]m [0;3

# Contextualized Word Embeddings From a Language Model (Like BERT)

In [23]:
from transformers import AutoModel, AutoTokenizer

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")
# Tokenize the sentence
tokens = tokenizer('Hello world', return_tensors='pt')

# Process the tokens
output = model(**tokens)[0]

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/a

In [24]:
output.shape

torch.Size([1, 4, 384])

In [26]:
for token in tokens['input_ids'][0]:
  print(tokenizer.decode(token))

[CLS]
Hello
 world
[SEP]


In [25]:
output

tensor([[[-3.4805,  0.0862, -0.1818,  ..., -0.0610, -0.3909,  0.3022],
         [ 0.1885,  0.3201, -0.2313,  ...,  0.3721,  0.2471,  0.8057],
         [ 0.2089,  0.5010, -0.0495,  ...,  1.2197, -0.2277,  0.8574],
         [-3.4277,  0.0635, -0.1426,  ...,  0.0658, -0.4358,  0.3826]]],
       dtype=torch.float16, grad_fn=<NativeLayerNormBackward0>)

# Text Embeddings (For Sentences and Whole Documents)

In [27]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Convert text to text embeddings
vector = model.encode("Best movie ever!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

... (3 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [28]:
vector.shape

(768,)